# 03 - Broadcast Joins y Window Functions

### Broadcast Hash Join
Al unir una tabla grande con una dimensional pequeña (equipos), `F.broadcast()` replica la tabla pequeña en todos los ejecutores, **eliminando el Shuffle por red**.


In [1]:
import sys
sys.path.append("..")
from src.config import get_spark_session
from src.etl.olympics_pipeline import DEPORTISTAS_SCHEMA, calcular_imc
import pyspark.sql.functions as F
from pyspark.sql.window import Window

spark = get_spark_session("03_Transformaciones")
df_dep = spark.read.schema(DEPORTISTAS_SCHEMA).option("header", "true").csv("../data/raw/deportista.csv")
df_eq = spark.read.option("header", "true").option("inferSchema", "true").csv("../data/raw/equipos.csv")

df_join = df_dep.transform(calcular_imc).join(F.broadcast(df_eq), on="equipo_id", how="inner")
df_join.select("nombre", "pais", "imc").show(5)


+--------------------+-----------+-----+
|              nombre|       pais|  imc|
+--------------------+-----------+-----+
|           A Dijiang|      China|24.69|
|            A Lamusi|      China|20.76|
| Gunnar Nielsen Aaby|    Denmark|23.51|
|Edgar Lindenau Aabye|    Denmark|25.66|
|Christine Jacoba ...|Netherlands|23.96|
+--------------------+-----------+-----+
only showing top 5 rows



### Window Functions
Cálculo de rankings dentro de cada partición sin colapsar registros:


In [2]:
ventana = Window.partitionBy("pais").orderBy(F.col("altura").desc())
df_rank = df_join.filter(F.col("altura").isNotNull()).withColumn("rank_altura", F.dense_rank().over(ventana))
df_rank.select("pais", "nombre", "altura", "rank_altura").show(10)


+-----------+--------------------+------+-----------+
|       pais|              nombre|altura|rank_altura|
+-----------+--------------------+------+-----------+
|      China|           A Dijiang| 180.0|          1|
|      China|            A Lamusi| 170.0|          2|
|    Denmark|Edgar Lindenau Aabye| 182.0|          1|
|    Denmark| Gunnar Nielsen Aaby| 175.0|          2|
|    Finland|Antti Frans Silas...| 178.0|          1|
|    Jamaica|          Usain Bolt| 195.0|          1|
|    Jamaica|Shelly-Ann Fraser...| 152.0|          2|
|Netherlands|Christine Jacoba ...| 185.0|          1|
|Netherlands| Cornelia Cor Aalten| 168.0|          2|
|     Norway|Einar Ferdinand M...| 176.0|          1|
+-----------+--------------------+------+-----------+
only showing top 10 rows

